# Setup do Ambiente (Unity Catalog)

Cria os schemas da Arquitetura Medalhão dentro do catalog `workspace`, já com descrição e
tags, e confere se os arquivos Parquet estão na landing zone.

Os Parquet são produzidos fora do Databricks por `scripts_auxiliares/extrair_dados.py`,
porque o **Free Edition bloqueia requisições HTTP de saída**. Nenhum notebook deste projeto
faz chamada de rede.

In [ ]:
# Parâmetros: usados pelo Job (Jobs & Pipelines) e com padrão para execução interativa.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("landing", "/Volumes/workspace/raw/files")

CATALOG = dbutils.widgets.get("catalog")
LANDING = dbutils.widgets.get("landing")

spark.sql(f"USE CATALOG {CATALOG}")

CAMADAS = {
    "bronze": "Camada Bronze: dados das fontes públicas como recebidos, em Delta, com metadados de ingestão. Nenhuma regra de negócio aplicada.",
    "silver": "Camada Silver: dados limpos, tipados, restritos à rede pública e agregados no grão município.",
    "gold": "Camada Gold: esquema estrela com 2 dimensões conformadas e 3 fatos, pronto para análise.",
}

for schema, descricao in CAMADAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema} COMMENT '{descricao}'")
    # IF NOT EXISTS não altera um schema que já existe; o COMMENT ON mantém a descrição em dia.
    spark.sql(f"COMMENT ON SCHEMA {CATALOG}.{schema} IS '{descricao}'")
    spark.sql(f"ALTER SCHEMA {CATALOG}.{schema} SET TAGS ('camada' = '{schema}', 'projeto' = 'mvp_educacao_social')")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## Verificação da landing zone

Confere se os 6 arquivos esperados foram enviados ao Volume. A execução deve falhar aqui,
e não lá na frente, se algum arquivo estiver faltando.

In [ ]:
ESPERADOS = [
    "municipios_ibge.parquet",
    "populacao_municipios.parquet",
    "bolsa_familia_brasil.parquet",
    "ideb_municipios.parquet",
    "censo_escolar_escolas.parquet",
    "dicionario_censo_escolar.parquet",
]

encontrados = {arquivo.name for arquivo in dbutils.fs.ls(LANDING)}
faltando = [nome for nome in ESPERADOS if nome not in encontrados]

display(dbutils.fs.ls(LANDING))
assert not faltando, f"Faltam arquivos na landing zone: {faltando}"
print(f"OK: os {len(ESPERADOS)} arquivos estão em {LANDING}")

# O volume de landing também recebe descrição no catálogo.
volume = ".".join(LANDING.strip("/").split("/")[1:4])
spark.sql(
    f"COMMENT ON VOLUME {volume} IS 'Landing zone: Parquet gerados fora do Databricks por "
    f"scripts_auxiliares/extrair_dados.py, porque o Free Edition bloqueia requisições HTTP de saída.'"
)